© 2026 Mechanome Bio. Proprietary and confidential.

# Cell and nuclear shape across cell types shared between datasets

**Kernel: Python** (needs `pandas`, `numpy`, `matplotlib`, `scipy`, `anndata`, `spatialdata` and the AWS command line tool `aws`; run on the SageMaker instance, which has access to the S3 buckets).

**What this notebook does**

1. **Copies the data in from S3** (cell 1): for each dataset in `DATASETS`, the zarr store (segmentation outlines for cells and nuclei) and the annotated AnnData (cell type labels). Anything already copied is skipped.
2. **Measures the shape of every cell and nucleus** (cells 2 and 3) with the same polygon code as the other notebooks: the six shape metrics, area and minor axis length, for the **cell outline** and the **nucleus outline** (names start with `cell_` or `nuclear_`). Cells are matched between the zarr and the AnnData by the part of the cell ID they share (the extra tile prefix MechanoMap adds is removed). The metrics are added to each dataset's AnnData.
3. **Divides every metric by the median of that metric in the cell's original dataset** (cell 3), so a value of 1.0 means "the dataset's median cell". This puts separately imaged and segmented datasets on one scale.
4. **Finds the cell types shared across datasets** (cell 4): a cell type is shared when it is present in at least `MINIMUM_DATASETS_PER_CELL_TYPE` datasets (default 3), where "present" means at least `MINIMUM_CELLS_PER_CELL_TYPE_IN_DATASET` cells. A **combined AnnData** of those cells is built and saved, with a `dataset` column recording each cell's original dataset.
5. **Figures** (cells 5 to 7), all for the median-normalised metrics, saved in `outputs/shared_cell_types/figures/`:
   - **Violin plots**: one per metric, one violin per shared cell type, all datasets pooled (the black line is the median). No statistical tests.
   - **Overlaid histograms**: one per metric, one step outline per cell type, drawn on the same bins; the vertical axis is the fraction of that cell type's cells per bin.
   - **Boxplots**: one per metric. Each point is the **median of one cell type in one dataset**, so the **datasets are the replicates** (n = number of datasets containing the cell type, **not** the number of cells). Every pair of cell types is compared with **Welch's t-test**, **Benjamini-Hochberg corrected** across the pairs in that figure. **Only significant comparisons are drawn; non-significant comparisons are not shown** (all p values are saved in each figure's `_metadata.json` and in `boxplot_statistics.csv`).

For the violins and histograms only, values outside the 0.1st to 99.9th percentile of each metric (across all cell types) are left out of the drawing; how many is stored in each figure's metadata. The boxplots and statistics use every cell.

| Input | What it holds |
|---|---|
| `inputs/zarrs/<dataset>` | zarr store with the cell and nucleus outlines (copied from S3) |
| `inputs/annotated_adatas/<dataset>.h5ad` | annotated AnnData with cell type labels (copied from S3) |

| Written by this notebook | What it holds |
|---|---|
| `outputs/shared_cell_types/prep/<dataset>_shape_metrics.csv` | saved shape measurements for each dataset (so the slow step runs once) |
| `outputs/shared_cell_types/prep/combined_shared_cell_types.h5ad` | combined AnnData of the shared cell types |
| `outputs/shared_cell_types/prep/combined_shape_metrics_per_cell.csv` | the per-cell table behind every figure |
| `outputs/shared_cell_types/prep/median_per_cell_type_per_dataset.csv` | the points in the boxplots |

© 2026 Mechanome Bio. Proprietary and confidential.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 1: imports, settings, and copying the data in from S3 storage.
# Everything you might want to change is in this cell.
import datetime
import itertools
import json
import platform
import re
import subprocess
import sys
from pathlib import Path

import numpy
import pandas
import anndata
import spatialdata
import matplotlib.pyplot as plt

# This notebook lives in nuclear_shape/notebooks, so the repository folder is one level up.
REPOSITORY_FOLDER = Path.cwd().parent
sys.path.insert(0, str(REPOSITORY_FOLDER / "src"))
import figure_style                      # Mechanome Bio house style (src/figure_style.py)
import nuclear_shape_metrics as metrics  # exact polygon measurements (src/nuclear_shape_metrics.py)

# Must be called before any figure is created: sets the shared font, font size and colours.
figure_style.apply_house_style()
PROPRIETARY_NOTICE = figure_style.PROPRIETARY_NOTICE

# ---------------------------------------------------------------------------------------------
# Which datasets to copy in and analyse. Add more names here to compare more datasets.
# ---------------------------------------------------------------------------------------------
DATASETS = [
    "pancreas-1",
    "lung-2",
    "colon-5",
]

# ---------------------------------------------------------------------------------------------
# Where the data lives in S3 (Amazon's cloud file storage), and where it is copied to.
# ---------------------------------------------------------------------------------------------
AWS_REGION = "eu-west-2"
# One zarr store (the segmentation outlines) per dataset: <ZARR_STORES_S3_FOLDER>/<dataset>
ZARR_STORES_S3_FOLDER = "s3://mechanome-cellpose-airr-lon-161012475647-eu-west-2/core/spt"
# One annotated AnnData file (cell types) per dataset: <ANNOTATED_ADATA_S3_FOLDER>/<dataset>/mb04/annotated_adata.h5ad
ANNOTATED_ADATA_S3_FOLDER = "s3://mechanome-mechanomap-airr-lon-161012475647-eu-west-2/core"
ANNOTATED_ADATA_FILE_PATH_IN_DATASET = "mb04/annotated_adata.h5ad"

# Local copies, on the SageMaker instance. The rest of the notebook only reads from here.
INPUT_FOLDER = Path("/home/ec2-user/SageMaker/inputs")
ZARR_FOLDER = INPUT_FOLDER / "zarrs"
ANNOTATED_ADATA_FOLDER = INPUT_FOLDER / "annotated_adatas"

# False = skip anything already copied in; True = copy everything again (use this if a
# previous copy was interrupted part way through).
COPY_AGAIN_FROM_S3 = False

# ---------------------------------------------------------------------------------------------
# What is inside the zarr stores and the annotated AnnData files.
# ---------------------------------------------------------------------------------------------
# Names of the outline layers inside each zarr store (same names as the Xenium zarr stores
# used in the other notebooks). A clear error lists the available layers if a name is wrong.
CELL_OUTLINE_LAYER_NAME = "cell_boundaries"
NUCLEAR_OUTLINE_LAYER_NAME = "nucleus_boundaries"

# Column of the annotated AnnData's obs table that holds the cell type label.
CELL_TYPE_COLUMN = "cell_type"

# How cells are matched between the zarr outlines and the AnnData:
# the cell ID in the AnnData is the cell ID in the zarr plus extra text (MechanoMap adds a tile
# prefix such as "R0C0_Y0_X0_"). This pattern is removed from the AnnData cell IDs so that only
# the part shared by both remains. Change it if the extra text in your IDs looks different.
ADATA_CELL_ID_COLUMN = None  # None = use the AnnData's row names; or give the name of an obs column
EXTRA_TEXT_IN_ADATA_CELL_ID_PATTERN = r"^R\d+C\d+_Y\d+_X\d+_"
# Stop with an error if fewer than this fraction of the outlines are matched to an AnnData cell.
MINIMUM_FRACTION_OF_OUTLINES_MATCHED = 0.5

# Cell type names must be spelled identically in every dataset to count as the same cell type.
# Use this to rename labels that differ, e.g. {"Fibroblasts": "Fibroblast", "T cell": "T_cell"}.
CELL_TYPE_NAME_FIXES = {}

# ---------------------------------------------------------------------------------------------
# Which cell types are compared
# ---------------------------------------------------------------------------------------------
# A cell type is "shared" when it is present in at least this many datasets...
MINIMUM_DATASETS_PER_CELL_TYPE = 3
# ...where "present" means at least this many cells of that type in the dataset (a median of a
# handful of cells is not a trustworthy replicate).
MINIMUM_CELLS_PER_CELL_TYPE_IN_DATASET = 30

# ---------------------------------------------------------------------------------------------
# Shape metrics
# ---------------------------------------------------------------------------------------------
# The six shape metrics measured on BOTH the cell outline and the nucleus outline, plus area
# and minor axis length. Every name starts with the outline it was measured on, e.g.
# "cell_solidity" or "nuclear_shape_index".
BASE_SHAPE_METRICS = ["solidity", "extent", "shape_index", "circularity", "perimeter", "axis_ratio",
                      "area", "minor_axis_length"]
CELL_SHAPE_METRICS = [f"cell_{name}" for name in BASE_SHAPE_METRICS]
NUCLEAR_SHAPE_METRICS = [f"nuclear_{name}" for name in BASE_SHAPE_METRICS]
SHAPE_METRICS = CELL_SHAPE_METRICS + NUCLEAR_SHAPE_METRICS

# Some cells have more than one nucleus outline; the largest one is kept (as in the prep notebook).

# True = measure the outlines again even if a saved table from an earlier run exists.
REMEASURE_OUTLINES = False

# ---------------------------------------------------------------------------------------------
# Plot settings
# ---------------------------------------------------------------------------------------------
# For the violins and histograms only: values outside this percentile range of each metric
# (taken across all cell types) are left out of the drawing so a few extreme cells do not
# stretch the axes. Set to (0, 100) to draw every cell. The boxplots and statistics always
# use every cell.
DISPLAY_PERCENTILE_RANGE = (0.1, 99.9)
NUMBER_OF_HISTOGRAM_BINS = 60

# Statistics for the boxplots (chosen by you): Welch's t-test between every pair of cell types
# (n = number of datasets), Benjamini-Hochberg corrected across the pairs of one figure.
STATISTICAL_TEST = "welch t-test"
MULTIPLE_COMPARISONS_CORRECTION = "benjamini-hochberg"
SIGNIFICANCE_THRESHOLD = 0.05  # adjusted p value below this is drawn; everything else is not shown

# ---------------------------------------------------------------------------------------------
# Output folders
# ---------------------------------------------------------------------------------------------
OUTPUT_FOLDER = REPOSITORY_FOLDER / "outputs" / "shared_cell_types"
PREP_FOLDER = OUTPUT_FOLDER / "prep"
FIGURE_FOLDER = OUTPUT_FOLDER / "figures"
for folder in (ZARR_FOLDER, ANNOTATED_ADATA_FOLDER, PREP_FOLDER, FIGURE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


def run_aws_command(arguments):
    """Run one AWS command line tool command and return its text output (stops with the AWS error message if it fails)."""
    completed = subprocess.run(["aws", "--region", AWS_REGION] + arguments, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"'aws {' '.join(arguments)}' failed:\n{completed.stderr}")
    return completed.stdout


def zarr_store_is_already_copied(zarr_path):
    """A complete zarr store has a top-level marker file ('.zgroup' in zarr v2, 'zarr.json' in v3)."""
    return (zarr_path / ".zgroup").exists() or (zarr_path / "zarr.json").exists()


# ---------------------------------------------------------------------------------------------
# Copy the data in from S3
# ---------------------------------------------------------------------------------------------
for dataset in DATASETS:
    zarr_destination = ZARR_FOLDER / dataset
    adata_destination = ANNOTATED_ADATA_FOLDER / f"{dataset}.h5ad"

    if zarr_store_is_already_copied(zarr_destination) and not COPY_AGAIN_FROM_S3:
        print(f"{dataset}: zarr store already copied, skipping")
    else:
        # "sync" copies a whole folder (a zarr store is a folder of many small files).
        run_aws_command(["s3", "sync", f"{ZARR_STORES_S3_FOLDER}/{dataset}", str(zarr_destination), "--only-show-errors"])
        if not zarr_store_is_already_copied(zarr_destination):
            raise FileNotFoundError(f"Nothing was copied for {ZARR_STORES_S3_FOLDER}/{dataset} - check the dataset name and S3 path.")
        print(f"{dataset}: zarr store copied")

    if adata_destination.exists() and not COPY_AGAIN_FROM_S3:
        print(f"{dataset}: annotated AnnData already copied, skipping")
    else:
        run_aws_command(["s3", "cp", f"{ANNOTATED_ADATA_S3_FOLDER}/{dataset}/{ANNOTATED_ADATA_FILE_PATH_IN_DATASET}",
                         str(adata_destination), "--only-show-errors"])
        print(f"{dataset}: annotated AnnData copied")

print("\ndatasets successfully copied over; the rest of the notebook reads from", INPUT_FOLDER)

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 2: helper functions used by the next cells.


def write_csv_with_notice(data_frame, path, **to_csv_options):
    """Write a CSV whose first line is the proprietary notice (read back with comment="#")."""
    with open(path, "w", encoding="utf-8", newline="") as output_file:
        output_file.write(f"# {PROPRIETARY_NOTICE}\n")
        data_frame.to_csv(output_file, **to_csv_options)


def read_outline_layer(spatial_data, layer_name, zarr_name):
    """
    One row per outline: the cell ID (stored as a column or as the index) and the outline polygon.
    Stops with a message listing the available layers if the layer name is wrong.
    """
    if layer_name not in spatial_data.shapes:
        raise KeyError(f"No '{layer_name}' layer in {zarr_name}; the shapes layers found are "
                       f"{list(spatial_data.shapes.keys())}. Change the layer names in cell 1.")
    outline_layer = spatial_data.shapes[layer_name]
    if "cell_id" in outline_layer.columns:
        outline_cell_ids = outline_layer["cell_id"].astype(str).to_numpy()
    else:
        outline_cell_ids = outline_layer.index.astype(str).to_numpy()
    return pandas.DataFrame({"cell_id": outline_cell_ids, "geometry": outline_layer.geometry.to_numpy()})


def measure_outlines(outlines, column_prefix):
    """
    Measure the shape metrics on every outline (same cleaning and same measurements as the prep
    notebook): keep the largest piece of each outline, drop invalid or zero-area outlines, and
    keep only the largest outline if one cell has several (e.g. two nuclei).

    Returns a table indexed by cell ID whose columns start with column_prefix, e.g. with the
    prefix "cell": cell_solidity, cell_extent, ... cell_area, cell_minor_axis_length.
    """
    outlines = outlines.copy()
    outlines["geometry"] = [metrics.keep_largest_piece(shape) for shape in outlines["geometry"]]
    is_usable = numpy.array([metrics.is_usable_polygon(shape) for shape in outlines["geometry"]], dtype=bool)
    print(f"    invalid / zero-area {column_prefix} outlines dropped: {(~is_usable).sum():,}")
    outlines = outlines[is_usable].copy()

    outlines["outline_area"] = [shape.area for shape in outlines["geometry"]]
    cells_with_several_outlines = outlines["cell_id"].duplicated(keep=False).sum()
    print(f"    {column_prefix} outlines belonging to a cell with more than one outline (largest kept): "
          f"{cells_with_several_outlines:,}")
    outlines = outlines.sort_values("outline_area", ascending=False).drop_duplicates("cell_id")

    measurement_rows = []
    for shape in outlines["geometry"]:
        measurements = metrics.measure_nucleus(shape)  # works on any cleaned polygon, cell or nucleus
        # equivalent_ellipse_axes returns (major axis length, minor axis length).
        measurements["minor_axis_length"] = metrics.equivalent_ellipse_axes(shape)[1]
        measurement_rows.append(measurements)
    measurement_table = pandas.DataFrame(measurement_rows, index=outlines["cell_id"].to_numpy())
    measurement_table = measurement_table.rename(columns={"nucleus_polygon_area": "area"})
    measurement_table = measurement_table.replace([numpy.inf, -numpy.inf], numpy.nan)
    measurement_table = measurement_table.dropna(subset=metrics.SHAPE_METRIC_NAMES + ["area", "minor_axis_length"])
    measurement_table = measurement_table[metrics.SHAPE_METRIC_NAMES + ["area", "minor_axis_length"]]
    measurement_table.columns = [f"{column_prefix}_{name}" for name in measurement_table.columns]
    return measurement_table


def measure_dataset_shapes(dataset):
    """
    Read the zarr store of one dataset and measure the cell and the nuclear outlines of every cell.
    Returns a table indexed by cell ID with all of SHAPE_METRICS (only cells with both outlines).
    The result is saved in prep/ so this slow step only runs once per dataset.
    """
    cache_path = PREP_FOLDER / f"{dataset}_shape_metrics.csv"
    if cache_path.exists() and not REMEASURE_OUTLINES:
        shape_table = pandas.read_csv(cache_path, comment="#", index_col="cell_id")
        shape_table.index = shape_table.index.astype(str)
        print(f"    shape metrics read from {cache_path.name} ({len(shape_table):,} cells)")
        return shape_table

    zarr_path = ZARR_FOLDER / dataset
    print(f"    measuring outlines in {zarr_path.name} (this can take a few minutes)")
    try:
        # Read only the outlines (skips the large images).
        spatial_data = spatialdata.read_zarr(zarr_path, selection=("shapes",))
    except TypeError:
        spatial_data = spatialdata.read_zarr(zarr_path)  # older spatialdata: no "selection" option

    cell_outlines = read_outline_layer(spatial_data, CELL_OUTLINE_LAYER_NAME, zarr_path.name)
    nuclear_outlines = read_outline_layer(spatial_data, NUCLEAR_OUTLINE_LAYER_NAME, zarr_path.name)
    cell_measurements = measure_outlines(cell_outlines, "cell")
    nuclear_measurements = measure_outlines(nuclear_outlines, "nuclear")

    # An inner join keeps only the cells that have BOTH a usable cell outline and a usable nucleus outline.
    shape_table = cell_measurements.join(nuclear_measurements, how="inner")[SHAPE_METRICS]
    shape_table.index.name = "cell_id"
    print(f"    cells with both a cell and a nuclear outline: {len(shape_table):,}")
    write_csv_with_notice(shape_table, cache_path, index=True)
    return shape_table


def read_cell_types(dataset):
    """
    Read the annotated AnnData of one dataset. Returns (adata, cell_types): the AnnData with its row
    names changed to the cell IDs used in the zarr store (the extra text removed), and a Series of
    cell type labels with the same row names.
    """
    adata = anndata.read_h5ad(ANNOTATED_ADATA_FOLDER / f"{dataset}.h5ad")
    if CELL_TYPE_COLUMN not in adata.obs.columns:
        raise KeyError(f"No '{CELL_TYPE_COLUMN}' column in {dataset}; the obs columns are {list(adata.obs.columns)}. "
                       "Change CELL_TYPE_COLUMN in cell 1.")
    if ADATA_CELL_ID_COLUMN is None:
        original_cell_ids = pandas.Series(adata.obs_names.astype(str), index=adata.obs_names)
    else:
        original_cell_ids = adata.obs[ADATA_CELL_ID_COLUMN].astype(str)
    # Remove the extra text so only the part shared with the zarr store remains.
    shared_cell_ids = original_cell_ids.str.replace(EXTRA_TEXT_IN_ADATA_CELL_ID_PATTERN, "", regex=True).to_numpy()
    adata.obs_names = shared_cell_ids

    number_duplicated = int(adata.obs_names.duplicated().sum())
    if number_duplicated:
        print(f"    cell IDs that are not unique after removing the extra text (first kept): {number_duplicated:,}")
        adata = adata[~adata.obs_names.duplicated()].copy()

    cell_types = adata.obs[CELL_TYPE_COLUMN].astype(object).where(adata.obs[CELL_TYPE_COLUMN].notna())
    cell_types = cell_types.map(lambda name: CELL_TYPE_NAME_FIXES.get(str(name), str(name)) if pandas.notna(name) else name)
    return adata, cell_types


def check_cell_ids_match(dataset, shape_cell_ids, adata_cell_ids):
    """Stop with a helpful message (showing example IDs from both sides) if the cell IDs do not line up."""
    matched = len(set(shape_cell_ids) & set(adata_cell_ids))
    fraction_matched = matched / max(len(shape_cell_ids), 1)
    print(f"    outlines matched to an AnnData cell: {matched:,} of {len(shape_cell_ids):,} ({fraction_matched:.1%})")
    if fraction_matched < MINIMUM_FRACTION_OF_OUTLINES_MATCHED:
        raise ValueError(
            f"Only {fraction_matched:.1%} of the {dataset} outlines match an AnnData cell. Example zarr cell IDs: "
            f"{list(shape_cell_ids[:3])}. Example AnnData cell IDs (after removing the extra text): "
            f"{list(adata_cell_ids[:3])}. Adjust EXTRA_TEXT_IN_ADATA_CELL_ID_PATTERN or ADATA_CELL_ID_COLUMN in cell 1.")


def relative_column_name(metric):
    """Name of the column holding a metric divided by its median in the original dataset."""
    return f"{metric}_relative_to_dataset_median"


print("helper functions defined")

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 3: for each dataset, measure the cell and nuclear shapes of every cell, attach them (and the
# cell type) to the dataset's AnnData, and divide each metric by that dataset's median.
#
# Dividing by the dataset's own median turns every metric into "how many times the typical
# cell of this dataset" (1.0 = the dataset median). This lets datasets that were imaged and
# segmented separately be compared on the same scale. The median is taken over ALL the cells
# of the dataset that could be measured, before any cell types are removed.

annotated_adatas = {}          # dataset name -> AnnData with the shape metrics in its obs table
median_per_dataset = {}        # dataset name -> Series: the median of each metric in that dataset

for dataset in DATASETS:
    print(f"\n========== {dataset} ==========")
    shape_table = measure_dataset_shapes(dataset)
    adata, cell_types = read_cell_types(dataset)
    check_cell_ids_match(dataset, shape_table.index.to_numpy(), adata.obs_names.to_numpy())

    # Keep the cells that have shape metrics AND a cell type label.
    cell_ids_to_keep = adata.obs_names.intersection(shape_table.index)
    adata = adata[cell_ids_to_keep].copy()
    cell_types = cell_types.loc[cell_ids_to_keep]
    shape_table = shape_table.loc[cell_ids_to_keep]
    has_label = cell_types.notna().to_numpy()
    print(f"    cells with shape metrics and an AnnData entry: {len(cell_ids_to_keep):,} "
          f"(without a cell type label, dropped: {(~has_label).sum():,})")
    adata, cell_types, shape_table = adata[has_label].copy(), cell_types[has_label], shape_table[has_label]

    # Median of each metric over the whole dataset, then every cell's value divided by it.
    dataset_medians = shape_table.median()
    if (dataset_medians <= 0).any():
        raise ValueError(f"{dataset}: a metric has a median of zero or less, so it cannot be divided by: "
                         f"{dataset_medians[dataset_medians <= 0].to_dict()}")
    median_per_dataset[dataset] = dataset_medians

    # Replace the AnnData's obs table with just what this analysis needs, so the datasets can be
    # combined later without clashing columns. The expression matrix (adata.X) is kept.
    new_obs = pandas.DataFrame(index=adata.obs_names)
    new_obs["dataset"] = dataset
    new_obs["cell_type"] = cell_types.to_numpy()
    for metric in SHAPE_METRICS:
        new_obs[metric] = shape_table[metric].to_numpy()
        new_obs[relative_column_name(metric)] = shape_table[metric].to_numpy() / dataset_medians[metric]
    adata = anndata.AnnData(X=adata.X, obs=new_obs, var=adata.var[[]].copy())
    annotated_adatas[dataset] = adata

    print(f"    cell types in {dataset}: " + ", ".join(
        f"{name} ({count:,})" for name, count in new_obs["cell_type"].value_counts().items()))

print("\nall datasets measured")

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 4: find the cell types shared across datasets and build the combined AnnData.

# ---- Which cell types are shared? ----------------------------------------------------------------
all_cells_obs = pandas.concat([adata.obs for adata in annotated_adatas.values()])
cells_per_dataset_and_type = all_cells_obs.groupby(["dataset", "cell_type"]).size().unstack(fill_value=0)
cells_per_dataset_and_type = cells_per_dataset_and_type.reindex(DATASETS)

# A cell type is "present" in a dataset if it has enough cells there to give a meaningful median.
is_present = cells_per_dataset_and_type >= MINIMUM_CELLS_PER_CELL_TYPE_IN_DATASET
number_of_datasets_per_type = is_present.sum(axis=0)
shared_types = number_of_datasets_per_type[number_of_datasets_per_type >= MINIMUM_DATASETS_PER_CELL_TYPE].index

if len(shared_types) < 2:
    raise ValueError(f"Only {len(shared_types)} cell type(s) are present in at least {MINIMUM_DATASETS_PER_CELL_TYPE} datasets "
                     f"(with at least {MINIMUM_CELLS_PER_CELL_TYPE_IN_DATASET} cells each), so there is nothing to compare. "
                     "Check the table above for cell type names that differ between datasets (use CELL_TYPE_NAME_FIXES), "
                     "or lower MINIMUM_DATASETS_PER_CELL_TYPE.")

print("cells per dataset (rows) and cell type (columns), shared types first:")
column_order = list(shared_types) + [name for name in cells_per_dataset_and_type.columns if name not in shared_types]
print(cells_per_dataset_and_type[column_order].to_string())
print(f"\nshared cell types (present in >= {MINIMUM_DATASETS_PER_CELL_TYPE} datasets with >= "
      f"{MINIMUM_CELLS_PER_CELL_TYPE_IN_DATASET} cells each): {len(shared_types)}")
print(number_of_datasets_per_type[shared_types].to_string())

# ---- Combined AnnData -------------------------------------------------------------------------
# Keep a cell only if its type is shared AND the type is present in that cell's own dataset
# (a type with just a few cells in one dataset is not used from that dataset).
datasets_where_present = {cell_type: set(is_present.index[is_present[cell_type]]) for cell_type in shared_types}
subsets = []
for dataset, adata in annotated_adatas.items():
    keep = numpy.array([cell_type in datasets_where_present and dataset in datasets_where_present[cell_type]
                        for cell_type in adata.obs["cell_type"]], dtype=bool)
    subset = adata[keep].copy()
    subset.obs_names = [f"{dataset}__{cell_id}" for cell_id in subset.obs_names]  # keeps cell IDs unique across datasets
    subsets.append(subset)

# "inner" keeps only the genes that every dataset has.
combined_adata = anndata.concat(subsets, join="inner", merge="same")
combined_adata.uns["proprietary_notice"] = PROPRIETARY_NOTICE
combined_adata.uns["description"] = (
    "Cells of the cell types shared across datasets. obs holds the cell and nuclear shape metrics of every cell "
    "and the same metrics divided by the median of that metric in the cell's original dataset "
    "(columns ending in _relative_to_dataset_median).")
combined = combined_adata.obs.copy()  # the per-cell table used by all the plots below

# Cell types ordered from most to fewest cells (the same order in every figure).
cell_counts_per_type = combined["cell_type"].value_counts()
CELL_TYPE_ORDER = list(cell_counts_per_type.index)
CELL_TYPE_DISPLAY_NAMES = {name: str(name).replace("_", " ") for name in CELL_TYPE_ORDER}  # "Ductal_epithelial" -> "Ductal epithelial"
cell_type_colours = dict(zip(CELL_TYPE_ORDER, figure_style.get_colours(len(CELL_TYPE_ORDER))))  # fixed per cell type in every histogram

print(f"\ncombined AnnData: {combined_adata.n_obs:,} cells x {combined_adata.n_vars:,} genes shared by all datasets")

# ---- Display range for the violins and histograms (on the median-normalised values) ----------
display_range_per_metric = {}
cells_left_out_per_metric = {}
for metric in SHAPE_METRICS:
    values = combined[relative_column_name(metric)]
    lower_limit, upper_limit = numpy.percentile(values, DISPLAY_PERCENTILE_RANGE)
    display_range_per_metric[metric] = (float(lower_limit), float(upper_limit))
    cells_left_out_per_metric[metric] = int((~values.between(lower_limit, upper_limit)).sum())

# ---- Median of each metric per cell type per dataset (the replicates for the boxplots) ------
# One value per dataset per cell type: the median over that dataset's cells of that type.
relative_columns = [relative_column_name(metric) for metric in SHAPE_METRICS]
dataset_medians_per_cell_type = combined.groupby(["cell_type", "dataset"])[relative_columns].median()
dataset_medians_per_cell_type = dataset_medians_per_cell_type.dropna()

# ---- Save the combined data and a record of how it was made ------------------------------------
combined_adata.write_h5ad(PREP_FOLDER / "combined_shared_cell_types.h5ad")
write_csv_with_notice(combined, PREP_FOLDER / "combined_shape_metrics_per_cell.csv", index=True, index_label="cell_id")
write_csv_with_notice(pandas.DataFrame(median_per_dataset).T.rename_axis("dataset"),
                      PREP_FOLDER / "median_of_each_metric_per_dataset.csv", index=True)
write_csv_with_notice(dataset_medians_per_cell_type.reset_index(),
                      PREP_FOLDER / "median_per_cell_type_per_dataset.csv", index=False)
with open(PREP_FOLDER / "combined_metadata.json", "w", encoding="utf-8") as metadata_file:
    json.dump({
        "proprietary_notice": PROPRIETARY_NOTICE,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "notebook": "notebooks/cell_shape_by_cell_type.ipynb",
        "datasets": DATASETS,
        "shared_cell_types": [str(name) for name in CELL_TYPE_ORDER],
        "minimum_datasets_per_cell_type": MINIMUM_DATASETS_PER_CELL_TYPE,
        "minimum_cells_per_cell_type_in_dataset": MINIMUM_CELLS_PER_CELL_TYPE_IN_DATASET,
        "cells_per_dataset_and_cell_type": json.loads(cells_per_dataset_and_type.to_json()),
        "normalisation": "each metric divided by the median of that metric over all measured cells of the original dataset",
        "number_of_cells": int(combined_adata.n_obs),
        "number_of_genes_shared_by_all_datasets": int(combined_adata.n_vars),
        "python_version": platform.python_version(),
        "package_versions": {"spatialdata": spatialdata.__version__, "anndata": anndata.__version__,
                             "numpy": numpy.__version__, "pandas": pandas.__version__},
    }, metadata_file, indent=2, ensure_ascii=False)
print("saved the combined data to", PREP_FOLDER.relative_to(REPOSITORY_FOLDER))

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 5: one violin plot figure per metric (one violin per shared cell type, all datasets pooled).


def values_inside_display_range(cell_type, metric):
    """Median-normalised values of one metric for one cell type, with the extreme values (outside the display range) removed."""
    lower_limit, upper_limit = display_range_per_metric[metric]
    values = combined.loc[combined["cell_type"] == cell_type, relative_column_name(metric)].to_numpy()
    return values[(values >= lower_limit) & (values <= upper_limit)]


def common_information_for_metadata(metric):
    """Facts stored in each figure's metadata file."""
    return {
        "datasets": DATASETS,
        "metric": metric,
        "measured_on": "nucleus outline" if metric.startswith("nuclear_") else "cell outline",
        "values_plotted": "metric divided by the median of that metric in the cell's original dataset (dimensionless)",
        "shared_cell_types": [str(name) for name in CELL_TYPE_ORDER],
        "cells_per_cell_type": {str(name): int(count) for name, count in cell_counts_per_type.items()},
        "display_percentile_range": list(DISPLAY_PERCENTILE_RANGE),
        "display_value_range": list(display_range_per_metric[metric]),
        "cells_left_out_of_drawing": cells_left_out_per_metric[metric],
        "source_files": ["prep/combined_shape_metrics_per_cell.csv"],
    }


def plot_violins(metric):
    x_axis_label = "Cell type"
    y_axis_label = f"{metric} relative to dataset median\n(dimensionless; line = median)"
    title = figure_style.make_title(metric, x_axis_label)

    values_per_cell_type = [values_inside_display_range(cell_type, metric) for cell_type in CELL_TYPE_ORDER]
    positions = numpy.arange(len(CELL_TYPE_ORDER))

    figure, axes = plt.subplots(figsize=(max(8.0, 1.0 * len(CELL_TYPE_ORDER) + 3.0), 5.5))
    violin_parts = axes.violinplot(values_per_cell_type, positions=positions, widths=0.85,
                                   showmeans=False, showmedians=True, showextrema=False)
    # The cell types are already named on the x axis, so every violin uses the single house colour.
    for violin_body in violin_parts["bodies"]:
        violin_body.set_facecolor(figure_style.DEFAULT_DARK_BLUE)
        violin_body.set_edgecolor(figure_style.DEFAULT_DARK_BLUE)
        violin_body.set_alpha(0.65)
    violin_parts["cmedians"].set_color("black")
    violin_parts["cmedians"].set_linewidth(1.5)

    axes.set_xticks(positions)
    axes.set_xticklabels([CELL_TYPE_DISPLAY_NAMES[name] for name in CELL_TYPE_ORDER], rotation=45, ha="right")
    axes.set_xlabel(x_axis_label)
    axes.set_ylabel(y_axis_label)
    axes.set_title(title)

    figure_style.save_figure(figure, str(FIGURE_FOLDER / f"violin_{metric}_by_cell_type"),
                             title, x_axis_label, y_axis_label,
                             extra_information=common_information_for_metadata(metric))
    plt.show()
    plt.close(figure)


for metric in SHAPE_METRICS:
    plot_violins(metric)

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 6: one overlaid histogram figure per metric (one coloured outline per shared cell type).


def plot_overlaid_histograms(metric):
    x_axis_label = f"{metric} relative to dataset median (dimensionless)"
    y_axis_label = "Fraction of cells of that type\nper bin (dimensionless)"
    title = figure_style.make_title("Fraction of cells", metric)

    # Every cell type uses exactly the same bins so the outlines can be compared directly.
    lower_limit, upper_limit = display_range_per_metric[metric]
    bin_edges = numpy.linspace(lower_limit, upper_limit, NUMBER_OF_HISTOGRAM_BINS + 1)

    figure, axes = plt.subplots(figsize=(11, 5.5))
    for cell_type in CELL_TYPE_ORDER:
        values = values_inside_display_range(cell_type, metric)
        # Dividing by the number of cells of THIS type turns counts into fractions, so cell
        # types with many more cells than others do not dominate the picture.
        counts_per_bin, _ = numpy.histogram(values, bins=bin_edges)
        fraction_per_bin = counts_per_bin / len(values)
        legend_text = f"{CELL_TYPE_DISPLAY_NAMES[cell_type]} (n = {cell_counts_per_type[cell_type]:,d})"
        # Outline only (no fill): with many overlapping cell types, filled bars blend into grey.
        axes.stairs(fraction_per_bin, bin_edges, fill=False, color=cell_type_colours[cell_type],
                    linewidth=1.6, label=legend_text)

    axes.set_xlim(lower_limit, upper_limit)
    axes.set_ylim(bottom=0)
    axes.set_xlabel(x_axis_label)
    axes.set_ylabel(y_axis_label)
    axes.set_title(title)
    # Legend on the right, outside the plotting area so it never hides any data.
    axes.legend(title="Cell type", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

    extra_information = common_information_for_metadata(metric)
    extra_information["number_of_histogram_bins"] = NUMBER_OF_HISTOGRAM_BINS
    figure_style.save_figure(figure, str(FIGURE_FOLDER / f"histogram_{metric}_by_cell_type"),
                             title, x_axis_label, y_axis_label, extra_information=extra_information)
    plt.show()
    plt.close(figure)


for metric in SHAPE_METRICS:
    plot_overlaid_histograms(metric)

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 7: one boxplot figure per metric, comparing cell types with the DATASETS as replicates.
#
# Each point is the median (of the median-normalised metric) of one cell type in ONE dataset, so
# the number of replicates (n) for a cell type is the number of datasets it is present in,
# not the number of cells. A box summarises those points. Cells of the same dataset are never
# treated as independent replicates.
#
# Statistics (per figure): Welch's t-test (compares two group means without assuming both groups
# have the same spread) between EVERY pair of cell types, then Benjamini-Hochberg correction
# (controls the false discovery rate, the expected share of false positives among the results
# called significant) across all the pairs in that figure. Only pairs with an adjusted p value
# below SIGNIFICANCE_THRESHOLD get a bracket; the other comparisons are not drawn (their p values are
# still saved in each figure's metadata file and in boxplot_statistics.csv).

all_statistics_rows = []  # one row per metric per pair of cell types, written to boxplot_statistics.csv at the end


def plot_boxplot(metric):
    column = relative_column_name(metric)

    # One list of per-dataset medians for each cell type, in the usual cell type order.
    values_by_cell_type = {}
    datasets_by_cell_type = {}
    for cell_type in CELL_TYPE_ORDER:
        medians = dataset_medians_per_cell_type.loc[cell_type, column]
        values_by_cell_type[CELL_TYPE_DISPLAY_NAMES[cell_type]] = medians.to_numpy(dtype=float)
        datasets_by_cell_type[CELL_TYPE_DISPLAY_NAMES[cell_type]] = list(medians.index)
    category_names = list(values_by_cell_type.keys())

    # ---- Statistics: Welch's t-test between every pair, Benjamini-Hochberg corrected ----------
    all_pairs = list(itertools.combinations(category_names, 2))
    # A t-test cannot be run when BOTH groups have exactly the same value in every replicate (zero spread).
    testable_pairs = [(first, second) for first, second in all_pairs
                      if values_by_cell_type[first].std() > 0 or values_by_cell_type[second].std() > 0]
    if len(testable_pairs) < len(all_pairs):
        print(f"  {metric}: {len(all_pairs) - len(testable_pairs)} pair(s) skipped (no spread in either group)")
    comparison_results = figure_style.run_comparisons(
        values_by_cell_type, testable_pairs, test_name=STATISTICAL_TEST, correction_name=MULTIPLE_COMPARISONS_CORRECTION)
    significant_results = [result for result in comparison_results if result["adjusted_p_value"] < SIGNIFICANCE_THRESHOLD]
    for result in comparison_results:
        all_statistics_rows.append({"metric": metric, **result})

    # ---- Draw ---------------------------------------------------------------------------------
    figure, axes = plt.subplots(figsize=(max(8.0, 1.0 * len(category_names) + 3.0), 6.0))
    positions = numpy.arange(len(category_names))
    box_parts = axes.boxplot([values_by_cell_type[name] for name in category_names], positions=positions,
                             widths=0.55, patch_artist=True, showfliers=False)
    for box in box_parts["boxes"]:
        box.set_facecolor(figure_style.DEFAULT_DARK_BLUE)
        box.set_alpha(0.45)
        box.set_edgecolor(figure_style.DEFAULT_DARK_BLUE)
    for line in box_parts["whiskers"] + box_parts["caps"]:
        line.set_color(figure_style.DEFAULT_DARK_BLUE)
    for line in box_parts["medians"]:
        line.set_color("black")
        line.set_linewidth(1.5)

    # Every dataset's median is drawn as a point (jittered sideways so they do not hide each other).
    random_generator = numpy.random.default_rng(0)  # fixed seed so the scatter is repeatable
    x_positions = {}
    highest_drawn_point = {}
    for position, name in enumerate(category_names):
        values = values_by_cell_type[name]
        axes.scatter(position + random_generator.uniform(-0.12, 0.12, size=len(values)), values,
                     s=24, color="white", edgecolor="black", linewidth=0.8, zorder=3)
        x_positions[name] = position
        highest_drawn_point[name] = float(values.max())

    all_values = numpy.concatenate(list(values_by_cell_type.values()))
    value_range = float(all_values.max() - all_values.min()) or 0.02
    axes.set_ylim(all_values.min() - 0.08 * value_range, all_values.max() + 0.08 * value_range)
    figure_style.add_significance_brackets(axes, x_positions, highest_drawn_point, significant_results)

    axes.set_xticks(positions)
    axes.set_xticklabels([f"{name} (n = {len(values_by_cell_type[name])})" for name in category_names],
                         rotation=45, ha="right")
    # The note about the unshown comparisons sits in the x label, together with the significance key.
    x_axis_label = ("Cell type (n = number of datasets)\n"
                    "Brackets: Welch's t-test, Benjamini-Hochberg adjusted p value: * < 0.05, ** < 0.01, *** < 0.001\n"
                    "Non-significant comparisons are not shown")
    y_axis_label = f"{metric} relative to dataset median\n(dimensionless; each point = median of one dataset)"
    title = figure_style.make_title(metric, "Cell type")
    axes.set_xlabel(x_axis_label)
    axes.set_ylabel(y_axis_label)
    axes.set_title(title)

    extra_information = common_information_for_metadata(metric)
    extra_information.pop("display_percentile_range"), extra_information.pop("display_value_range")
    extra_information.pop("cells_left_out_of_drawing")  # the boxplots use every cell, nothing is left out
    extra_information.update({
        "points_plotted": "median over the cells of one cell type in one dataset, of the metric divided by its dataset median",
        "replicates": "datasets (n = number of datasets containing the cell type)",
        "datasets_per_cell_type": datasets_by_cell_type,
        "values_per_cell_type": {name: [float(value) for value in values] for name, values in values_by_cell_type.items()},
        "comparisons": "every pair of cell types",
        "significance_threshold_on_adjusted_p": SIGNIFICANCE_THRESHOLD,
        "only_significant_comparisons_are_drawn": True,
        "number_of_comparisons_made": len(comparison_results),
        "number_of_comparisons_drawn": len(significant_results),
        "all_comparisons_including_non_significant": comparison_results,
        "correction_statement": figure_style.describe_correction(MULTIPLE_COMPARISONS_CORRECTION, len(comparison_results)),
        "source_files": ["prep/median_per_cell_type_per_dataset.csv"],
    })
    # Only the drawn (significant) comparisons go in "statistical_comparisons"; the full list, including the
    # non-significant ones that are not drawn, is in "all_comparisons_including_non_significant" above.
    if not significant_results:
        extra_information["statistical_comparisons"] = (
            "Tests were run for every pair of cell types; none were significant, so no brackets are drawn.")
    figure_style.save_figure(figure, str(FIGURE_FOLDER / f"boxplot_{metric}_by_cell_type"),
                             title, x_axis_label, y_axis_label,
                             comparison_results=significant_results or None,
                             extra_information=extra_information)
    plt.show()
    plt.close(figure)


for metric in SHAPE_METRICS:
    plot_boxplot(metric)

# ---- All the statistics in one table --------------------------------------------------------------
statistics_table = pandas.DataFrame(all_statistics_rows, columns=["metric", "first_category", "second_category", "test", "correction",
                                                                 "raw_p_value", "adjusted_p_value", "significance_label"])
statistics_table = statistics_table.drop(columns=["test", "correction"])
statistics_table["test"] = STATISTICAL_TEST
statistics_table["correction"] = MULTIPLE_COMPARISONS_CORRECTION
write_csv_with_notice(statistics_table, FIGURE_FOLDER / "boxplot_statistics.csv", index=False)
print(f"\nTest: {STATISTICAL_TEST}; correction: {MULTIPLE_COMPARISONS_CORRECTION} "
      "(applied separately to the pairs of cell types in each figure)")
print("\nsignificant comparisons (adjusted p < {:g}):".format(SIGNIFICANCE_THRESHOLD))
significant_table = statistics_table[statistics_table["adjusted_p_value"] < SIGNIFICANCE_THRESHOLD]
print(significant_table[["metric", "first_category", "second_category", "raw_p_value", "adjusted_p_value",
                         "significance_label"]].to_string(index=False) if len(significant_table) else "none")

### Output files

For each metric `<metric>` (`cell_` or `nuclear_` followed by `solidity`, `extent`, `shape_index`, `circularity`, `perimeter`, `axis_ratio`, `area`, `minor_axis_length`), in `outputs/shared_cell_types/figures/`:

- `violin_<metric>_by_cell_type.png / .svg / _metadata.json`
- `histogram_<metric>_by_cell_type.png / .svg / _metadata.json`
- `boxplot_<metric>_by_cell_type.png / .svg / _metadata.json` (statistics: Welch's t-test between every pair of cell types, Benjamini-Hochberg corrected per figure; only significant comparisons drawn)
- `boxplot_statistics.csv`: raw and adjusted p value of every comparison for every metric, including the non-significant ones that are not drawn

© 2026 Mechanome Bio. Proprietary and confidential.